# Signal Processing

screamer includes causal signal-processing filters: Butterworth low-pass,
Hull Moving Average, detrending, and more.  Because every filter is
**causal** — it uses only past and present samples, never future ones —
the batch and live results are identical, and filters compose naturally
with screamer's other operators.  This makes them safe to use in
production streaming pipelines, not just offline analysis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from screamer import Butter, HullMA, Detrend

rng = np.random.default_rng(3)
t = np.linspace(0, 4 * np.pi, 500)
clean = np.sin(t)
noisy = clean + 0.4 * rng.standard_normal(t.size)

## Low-pass filtering with `Butter`

`Butter(order, cutoff_freq)` implements a causal Butterworth filter.
`cutoff_freq` is normalised to the Nyquist frequency (0 < f < 0.5).
Higher order → sharper roll-off but more phase delay (all causal — no
zero-phase post-processing).

In [ ]:
lp = Butter(4, 0.05)(noisy)     # 4th-order low-pass, cutoff 0.05

print("Butter(4, 0.05) output range:", np.nanmin(lp).round(3), "–", np.nanmax(lp).round(3))

## Smoothing with `HullMA`

The Hull Moving Average reduces lag compared to a simple rolling average
while staying causal.  It does this by combining a short-window WMA and a
long-window WMA to compensate for the smoothing delay.

In [ ]:
hull = HullMA(20)(noisy)

print("HullMA(20) output range:", np.nanmin(hull).round(3), "–", np.nanmax(hull).round(3))

## Detrending with `Detrend`

`Detrend(window)` subtracts a rolling linear trend estimated over the last
`window` samples.  The result is stationary around zero even when the
original series has a strong drift.

In [ ]:
detr = Detrend(50)(100 + t + noisy)   # remove a rolling trend from a drifting series

print("Detrend(50) mean :", np.nanmean(detr).round(4))
print("Detrend(50) std  :", np.nanstd(detr).round(4))

## Comparison plot

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharex=True)

ax[0].plot(noisy, lw=0.4, alpha=0.6, label="noisy signal")
ax[0].plot(clean, lw=0.8, color="k", alpha=0.4, label="clean sin(t)")
ax[0].plot(lp, label="Butter(4, 0.05)")
ax[0].plot(hull, label="HullMA(20)")
ax[0].legend(loc="upper right", fontsize=8)
ax[0].set_title("Low-pass filters on a noisy sine wave")

ax[1].plot(detr, label="Detrend(50)")
ax[1].axhline(0, color="k", lw=0.4)
ax[1].legend(loc="upper right", fontsize=8)
ax[1].set_title("Detrended signal (drift removed)")

plt.tight_layout()

## Causality: batch == live

Filters are just functors — the scalar loop and the array path give the same
output.  This is a hard guarantee: any filter that differed between batch and
live would have broken causality.

In [ ]:
# Verify Butter filter is causal (batch == live scalar loop)
b = Butter(4, 0.05)
live_lp = np.array([b(v) for v in noisy])

np.testing.assert_allclose(lp, live_lp, equal_nan=True)
print("Butter batch == live ✓")

**Takeaways**

- All screamer filters are causal: no lookahead, no zero-phase trick, same
  result in backtest and live.
- `Butter(order, cutoff)` is a standard Butterworth IIR filter; `HullMA`
  smooths with less lag than a plain rolling mean.
- `Detrend(window)` removes a rolling linear trend, making drifting series
  stationary without differencing.
- Filters are functors: they compose with every other screamer operator and
  work identically in batch and scalar-loop (live) mode.